# Water–isobutanol phase discovery

**Evidence: numerical verification.** Compare declared liquid phases with native phase discovery.


The canonical input is the neutral water/isobutanol subset of
`khudaida-2026-figure-2-electrolyte-lle/7`, at 293.15 K and 100,000 Pa.
Khudaida et al. (2026), Tables 2, 5–7
([paper](https://doi.org/10.1021/acs.jced.5c00780)), and the packet's cited
association rules own the inputs. Version 7 retains the Wolbach–Sandler
unlike association size factor. The feed is 0.6 mol water + 0.4 mol isobutanol.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from epcsaft import (
    Parameters,
    Mixture,
)
from epcsaft import equilibrium as eq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "ARCHITECTURE.yaml").is_file()
)
PACKETS = ROOT / "data" / "packets"


def equilibrium_values(declared_state):
    """Keep the returned physical state and the equation residuals together."""
    assert declared_state.success, declared_state.message
    assert declared_state.requested_tolerance_met
    assert declared_state.classified_residuals
    residual = max(map(abs, declared_state.classified_residuals), default=0)
    assert residual <= 1e-10
    return dict(
        temperature_K=declared_state.temperature,
        pressure_Pa=declared_state.pressure,
        amounts_mol=declared_state.amounts,
        mole_fractions=declared_state.mole_fractions,
        molar_densities_mol_m3=declared_state.molar_densities,
        residual=residual,
        linear_solver=declared_state.linear_solver,
    )


## Declare a split, then allow discovery

The two liquid guesses are (0.999, 0.001) and (0.3, 0.7).
`Discover` delegates finite phase trials to the Engine. Match phases by water
fraction, since returned phase ordinals need not match. Require endpoint
agreement within 10⁻⁹, component closure within 10⁻¹⁰ mol, and classified
equation residuals at most 10⁻¹⁰. Keep the tangent-plane trial evidence visible.


In [2]:
model = Mixture(
    Parameters.from_json(
        PACKETS / "khudaida-2026-figure-2-electrolyte-lle/7/parameters/parameters.json",
        components=["water", "isobutanol"],
    )
)
feed = eq.Amounts({"water": 0.6, "isobutanol": 0.4})
problem = eq.Problem(
    T=293.15,
    P=100000.0,
    feed=feed,
    phases=[
        eq.Phase("Lw", kind="liquid", composition_guess=(0.999, 0.001)),
        eq.Phase("Lo", kind="liquid", composition_guess=(0.3, 0.7)),
    ],
)
declared_state = eq.solve_equilibrium(model, problem)
report = eq.stability_test(eq.compile_problem(model, problem), declared_state)
discovered_state = eq.solve_equilibrium(
    model, eq.Problem(T=293.15, P=100000.0, feed=feed, phases=eq.Discover())
)
assert (
    max(
        (
            abs(a - b)
            for a, b in zip(
                sorted(declared_state.mole_fractions[::2]),
                sorted(discovered_state.mole_fractions[::2]),
            )
        )
    )
    < 1e-09
)
values = dict(
    declared=equilibrium_values(declared_state),
    discovered=equilibrium_values(discovered_state),
    trials=len(report.trials),
    message=report.message,
    completed=report.completed,
)
for solved in (declared_state, discovered_state):
    balances = [
        sum((solved.amounts[j] * solved.mole_fractions[2 * j + i] for j in range(2)))
        - amount
        for i, amount in enumerate((0.6, 0.4))
    ]
    assert max(map(abs, balances)) <= 1e-10
    equilibrium_values(solved)
values["component_balance_mol"] = balances
values["unresolved_trials"] = sum(
    ("unresolved" in trial.evidence.lower() for trial in report.trials)
)
values["trial_evidence"] = [trial.evidence for trial in report.trials]
assert report.completed and values["unresolved_trials"] == 0
display(
    pd.DataFrame(
        {
            "Declared water fraction / 1": declared_state.mole_fractions[::2],
            "Discovered water fraction / 1": sorted(
                discovered_state.mole_fractions[::2], reverse=True
            ),
            "Declared amount / mol": declared_state.amounts,
        }
    )
)
declared_order = sorted(
    range(2), key=lambda phase: declared_state.mole_fractions[2 * phase]
)
discovered_order = sorted(
    range(2), key=lambda phase: discovered_state.mole_fractions[2 * phase]
)
for declared, discovered in zip(declared_order, discovered_order):
    assert (
        abs(declared_state.amounts[declared] - discovered_state.amounts[discovered])
        < 1e-09
    )
    assert (
        max(
            (
                abs(
                    declared_state.mole_fractions[2 * declared + i]
                    - discovered_state.mole_fractions[2 * discovered + i]
                )
                for i in range(2)
            )
        )
        < 1e-09
    )


,Declared water fraction / 1,Discovered water fraction / 1,Declared amount / mol
0,0.994375,0.994375,0.285318
1,0.442556,0.442556,0.714682


In [3]:
display(pd.DataFrame(
    [values["declared"], values["discovered"]], index=["Declared", "Discovered"]
).rename(columns={
            "temperature_K": "Temperature / K",
            "pressure_Pa": "Pressure / Pa",
            "amounts_mol": "Phase amounts / mol",
            "mole_fractions": "Mole fractions / 1",
            "molar_densities_mol_m3": "Phase densities / mol m^-3",
            "residual": "Maximum classified residual / 1",
            "linear_solver": "Linear solver",
        }))
display(pd.DataFrame({
    "Component": ["water", "isobutanol"],
    "Component balance error / mol": values["component_balance_mol"],
}))
display(pd.DataFrame([{
    "Trials / 1": values["trials"],
    "Unresolved trials / 1": values["unresolved_trials"],
    "Completed": values["completed"],
    "Message": values["message"],
}]))
display(pd.DataFrame({"Trial evidence": values["trial_evidence"]}))


,Temperature / K,Pressure / Pa,Phase amounts / mol,Mole fractions / 1,Phase densities / mol m^-3,Maximum classified residual / 1,Linear solver
Declared,293.15,100000.0,"[0.2853182931107769, 0.7146817068892232]","[0.9943748683039002, 0.005625131696099895, 0.4425559753585558, 0.5574440246414442]","[54198.3600462764, 16912.29929663858]",2.073248e-14,
Discovered,293.15,100000.0,"[0.7146817068891981, 0.285318293110802]","[0.4425559753585365, 0.5574440246414635, 0.9943748683038999, 0.005625131696100095]","[16912.299296638146, 54198.36004627646]",1.937552e-14,mumps


,Component,Component balance error / mol
0,water,0.000000e+00
1,isobutanol,5.551115e-17


,Trials / 1,Unresolved trials / 1,Completed,Message
0,7,0,True,no witness in the recorded trial set; unresolved trials 0


,Trial evidence
0,no_resolved_descent
1,no_resolved_descent
2,no_resolved_descent
3,no_resolved_descent
4,no_resolved_descent
5,no_resolved_descent
6,no_resolved_descent


## Meaning and limits


No witness found in a finite trial set is finite-search evidence. It is not
a global stability certificate. This neutral subset does not compare measured
salt-containing tie-lines, and no experimental uncertainty is estimated here.
